# Lesson 03 — DCT Basics

## Why
DCT (Discrete Cosine Transform) is what JPEG compression uses.
Understanding DCT means understanding why JPEG images look blocky when compressed too hard.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import dctn, idctn

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)

# Apply DCT to 8x8 blocks (like JPEG)
def dct_compress(gray, keep_percent=0.1):
    h, w     = gray.shape
    result   = np.zeros_like(gray)
    block_sz = 8

    for i in range(0, h-block_sz+1, block_sz):
        for j in range(0, w-block_sz+1, block_sz):
            block     = gray[i:i+block_sz, j:j+block_sz]
            dct_block = dctn(block, norm='ortho')
            # Zero out small coefficients (compression)
            thresh    = np.percentile(np.abs(dct_block), 100*(1-keep_percent))
            dct_block[np.abs(dct_block) < thresh] = 0
            result[i:i+block_sz, j:j+block_sz] = idctn(dct_block, norm='ortho')

    return np.clip(result, 0, 255).astype(np.uint8)

compressed_10 = dct_compress(gray, keep_percent=0.10)  # keep 10% of DCT coefficients
compressed_02 = dct_compress(gray, keep_percent=0.02)  # keep 2% (heavy compression)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, im, t in zip(axes, [gray, compressed_10, compressed_02],
    ['Original', '10% DCT coefficients kept', '2% DCT coefficients (heavy JPEG artifacts)']):
    ax.imshow(im.astype(np.uint8), cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('DCT compression — this is what JPEG does internally', fontsize=12)
plt.show()

## Key Takeaway
JPEG works by computing DCT on 8×8 blocks and discarding small coefficients.
The blockiness (JPEG artifacts) comes from throwing away too many coefficients.